In [1]:
import os
os.environ["POLARS_MAX_THREADS"] = "4"

import polars as pl

In [2]:
from pathlib import Path

from open_icu import OpenICUProject, ExtractionStep, ConceptStep#, ShardingStep
from open_icu.logging import configure_logging

configure_logging(level="DEBUG")

config_path = Path.cwd() / "config"
project_path = Path.cwd() / "output" / "project"
with OpenICUProject(project_path) as project:

    extraction_step = ExtractionStep.load(project, config_path / "extraction.yml")
    extraction_step.run()

    concept_step = ConceptStep.load(project, config_path / "concept.yml")
    concept_step.run()

    # sharding_step = ShardingStep.load(project, config_path / "sharding.yml")
    # sharding_step.run()

2026-05-29 11:58:00 [DEBUG] open_icu.storage.base: Initializing storage at /home/q039tl/OpenICU.example/output/project with overwrite=False
2026-05-29 11:58:00 [DEBUG] open_icu.steps.base.step: Step 'extraction': overwrite=False, workspace_exists=True, dataset_exists=True, skip=True
2026-05-29 11:58:00 [INFO] open_icu.steps.base.step: Running step 'extraction'
2026-05-29 11:58:00 [DEBUG] open_icu.steps.base.step: Step 'extraction': setting up config
2026-05-29 11:58:00 [DEBUG] open_icu.steps.base.step: Loading config file ../OpenICU/config/dataset/eicu-crd/2.0/dataset (overwrite=False)
2026-05-29 11:58:00 [DEBUG] open_icu.config.registry: Loading configs from ../OpenICU/config/dataset/eicu-crd/2.0/dataset (overwrite=False)
2026-05-29 11:58:00 [DEBUG] open_icu.config.registry: Loaded configuration openicu.config.dataset.eicu-crd.2.0.respiratorycharting from ../OpenICU/config/dataset/eicu-crd/2.0/dataset/respiratorycharting.yml
2026-05-29 11:58:00 [DEBUG] open_icu.config.registry: Loaded

In [ ]:
old_regex = "(225798//Vancomycin|225837//Acyclovir|225838//Ambisome|225840//Amikacin|225842//Ampicillin|225843//Ampicillin/Sulbactam \(Unasyn\)|225844//Atovaquone|225845//Azithromycin|225847//Aztreonam|225848//Caspofungin|225850//Cefazolin|225851//Cefepime|225853//Ceftazidime|225855//Ceftriaxone|225857//Chloroquine|225859//Ciprofloxacin|225860//Clindamycin|225862//Colistin|225863//Daptomycin|225865//Doxycycline|225866//Erythromycin|225868//Ethambutol|225869//Fluconazole|225871//Foscarnet|225873//Gancyclovir|225875//Gentamicin|225876//Imipenem/Cilastatin|225877//Isoniazid|225879//Levofloxacin|225881//Linezolid|225882//Mefloquine|225883//Meropenem|225884//Metronidazole|225885//Micafungin|225886//Moxifloxacin|225888//Nafcillin|225889//Oxacillin|225890//Penicillin G potassium|225892//Piperacillin|225893//Piperacillin/Tazobactam \(Zosyn\)|225895//Pyrazinamide|225896//Quinine|225897//Ribavirin|225898//Rifampin|225899//Bactrim \(SMX/TMP\)|225900//Dalfopristin/Quinupristin \(Synercid\)|225902//Tobramycin|225903//Valgancyclovir|225905//Voriconazole|227691//Keflex|228003//Tamiflu)"

In [4]:
new_regex = "(225798(//.*)?|225837(//.*)?|225838(//.*)?|225840(//.*)?|225842(//.*)?|225843(//.*)?|225844(//.*)?|225845(//.*)?|225847(//.*)?|225848(//.*)?|225850(//.*)?|225851(//.*)?|225853(//.*)?|225855(//.*)?|225857(//.*)?|225859(//.*)?|225860(//.*)?|225862(//.*)?|225863(//.*)?|225865(//.*)?|225866(//.*)?|225868(//.*)?|225869(//.*)?|225871(//.*)?|225873(//.*)?|225875(//.*)?|225876(//.*)?|225877(//.*)?|225879(//.*)?|225881(//.*)?|225882(//.*)?|225883(//.*)?|225884(//.*)?|225885(//.*)?|225886(//.*)?|225888(//.*)?|225889(//.*)?|225890(//.*)?|225892(//.*)?|225893(//.*)?|225895(//.*)?|225896(//.*)?|225897(//.*)?|225898(//.*)?|225899(//.*)?|225900(//.*)?|225902(//.*)?|225903(//.*)?|225905(//.*)?|227691(//.*)?|228003(//.*)?)"

In [5]:
import polars as pl

cols = ["subject_id", "time", "code", "numeric_value", "text_value", "hadm_id", "stay_id"]
lf1 = pl.scan_parquet("/home/q039tl/OpenICU.example/output/project/workspace/extraction/mimic-iv/3.1/inputevents/INFUSION_START.parquet").select(cols)
lf2 = pl.scan_parquet("/home/q039tl/OpenICU.example/output/project/workspace/extraction/mimic-iv/3.1/inputevents/INFUSION_END.parquet").select(cols)

In [6]:
lf1_old = lf1.filter(pl.col("code").str.contains(old_regex))
lf2_old = lf2.filter(pl.col("code").str.contains(old_regex))

lf_old = pl.concat([lf1_old, lf2_old], how="vertical_relaxed",)

In [7]:
df_old = lf_old.collect().sort(cols)
df_old

subject_id,time,code,numeric_value,text_value,hadm_id,stay_id
i64,datetime[μs],str,f32,str,i64,i64
10000690,2150-11-03 12:33:00,"""mimic-iv//inputevents//225798/…",null,null,25860671,37081114
10000690,2150-11-03 12:34:00,"""mimic-iv//inputevents//225798/…",null,"""FinishedRunning""",25860671,37081114
10000690,2150-11-03 19:00:00,"""mimic-iv//inputevents//225851/…",null,null,25860671,37081114
10000690,2150-11-03 19:01:00,"""mimic-iv//inputevents//225851/…",null,"""FinishedRunning""",25860671,37081114
10000690,2150-11-04 13:15:00,"""mimic-iv//inputevents//225798/…",null,null,25860671,37081114
…,…,…,…,…,…,…
19999840,2164-09-17 08:08:00,"""mimic-iv//inputevents//225798/…",null,"""FinishedRunning""",21033226,38978960
19999840,2164-09-17 09:32:00,"""mimic-iv//inputevents//225837/…",null,null,21033226,38978960
19999840,2164-09-17 09:33:00,"""mimic-iv//inputevents//225837/…",null,"""FinishedRunning""",21033226,38978960


In [8]:
lf1_new = lf1.filter(pl.col("code").str.contains(new_regex))
lf2_new = lf2.filter(pl.col("code").str.contains(new_regex))

lf_new = pl.concat([lf1_new, lf2_new], how="vertical_relaxed",)

In [9]:
df_new = lf_new.collect().sort(cols)
df_new

subject_id,time,code,numeric_value,text_value,hadm_id,stay_id
i64,datetime[μs],str,f32,str,i64,i64
10000690,2150-11-03 12:33:00,"""mimic-iv//inputevents//225798/…",null,null,25860671,37081114
10000690,2150-11-03 12:34:00,"""mimic-iv//inputevents//225798/…",null,"""FinishedRunning""",25860671,37081114
10000690,2150-11-03 19:00:00,"""mimic-iv//inputevents//225851/…",null,null,25860671,37081114
10000690,2150-11-03 19:01:00,"""mimic-iv//inputevents//225851/…",null,"""FinishedRunning""",25860671,37081114
10000690,2150-11-04 13:15:00,"""mimic-iv//inputevents//225798/…",null,null,25860671,37081114
…,…,…,…,…,…,…
19999840,2164-09-17 08:08:00,"""mimic-iv//inputevents//225798/…",null,"""FinishedRunning""",21033226,38978960
19999840,2164-09-17 09:32:00,"""mimic-iv//inputevents//225837/…",null,null,21033226,38978960
19999840,2164-09-17 09:33:00,"""mimic-iv//inputevents//225837/…",null,"""FinishedRunning""",21033226,38978960


In [10]:
len(lf1_old.collect())

616330

In [11]:
len(lf1_new.collect())

616330

In [12]:
len(lf2_old.collect())

616330

In [13]:
len(lf2_new.collect())

616330

In [14]:
lf1_only_in_new = lf1_new.select("code").join(
    lf1_old.select("code"), on=["code"],
    how="anti"
)

In [15]:
df = lf1_only_in_new.head().collect()

In [16]:
for row in df.rows():
    print(row)

In [17]:
df.filter(pl.col("code").str.contains(old_regex))

code
str


In [18]:
old_regex

'(225798//Vancomycin|225837//Acyclovir|225838//Ambisome|225840//Amikacin|225842//Ampicillin|225843//Ampicillin/Sulbactam \\(Unasyn\\)|225844//Atovaquone|225845//Azithromycin|225847//Aztreonam|225848//Caspofungin|225850//Cefazolin|225851//Cefepime|225853//Ceftazidime|225855//Ceftriaxone|225857//Chloroquine|225859//Ciprofloxacin|225860//Clindamycin|225862//Colistin|225863//Daptomycin|225865//Doxycycline|225866//Erythromycin|225868//Ethambutol|225869//Fluconazole|225871//Foscarnet|225873//Gancyclovir|225875//Gentamicin|225876//Imipenem/Cilastatin|225877//Isoniazid|225879//Levofloxacin|225881//Linezolid|225882//Mefloquine|225883//Meropenem|225884//Metronidazole|225885//Micafungin|225886//Moxifloxacin|225888//Nafcillin|225889//Oxacillin|225890//Penicillin G potassium|225892//Piperacillin|225893//Piperacillin/Tazobactam \\(Zosyn\\)|225895//Pyrazinamide|225896//Quinine|225897//Ribavirin|225898//Rifampin|225899//Bactrim \\(SMX/TMP\\)|225900//Dalfopristin/Quinupristin \\(Synercid\\)|225902//Tob

# TODO: oxygen_saturation

In [19]:
cols2 = ["subject_id", "time", "code", "numeric_value", "text_value"]

In [28]:
pl.read_parquet("/home/q039tl/OpenICU.example/output/project/workspace/concept/oxygen_saturation/1.0.0/mimic-iv.parquet").sort(cols2).describe()

statistic,subject_id,time,code,numeric_value,text_value,dataset,table
str,f64,str,str,f64,str,str,str
"""count""",9.377702e6,"""9377702""","""9377702""",9.377702e6,"""9377702""","""9377702""","""9377702"""
"""null_count""",0.0,"""0""","""0""",0.0,"""0""","""0""","""0"""
"""mean""",1.5015e7,"""2153-09-25 17:39:57.878211""",null,102.581551,null,null,null
"""std""",2.8928e6,null,null,7334.117676,null,null,null
"""min""",1.0000032e7,"""2110-01-11 12:42:00""","""oxygen_saturation//%""",-951234.0,"""-1""","""mimic-iv""","""chartevents"""
"""25%""",1.251701e7,"""2133-11-19 10:00:00""",null,94.0,null,null,null
"""50%""",1.5032392e7,"""2153-07-29 07:00:00""",null,97.0,null,null,null
"""75%""",1.7521905e7,"""2173-10-15 08:00:00""",null,99.0,null,null,null
"""max""",1.9999987e7,"""2214-07-26 16:00:00""","""oxygen_saturation//%""",9.9e6,"""999""","""mimic-iv""","""chartevents"""


In [29]:
pl.read_parquet("/home/q039tl/OpenICU.example/output/project/workspace/concept-cur/oxygen_saturation/1.0.0/mimic-iv.parquet").sort(cols2).describe()

statistic,subject_id,time,code,numeric_value,text_value,dataset,table
str,f64,str,str,f64,str,str,str
"""count""",9.617261e6,"""9617261""","""9617261""",9.617044e6,"""9617044""","""9617261""","""9617261"""
"""null_count""",0.0,"""0""","""0""",217.0,"""217""","""0""","""0"""
"""mean""",1.5016e7,"""2153-09-27 20:22:47.676141""",null,102.109741,null,null,null
"""std""",2.8929e6,null,null,7242.280762,null,null,null
"""min""",1.0000032e7,"""2109-07-29 11:21:00""","""oxygen_saturation//%""",-951234.0,"""-1""","""mimic-iv""","""chartevents"""
"""25%""",1.2516814e7,"""2133-11-24 03:00:00""",null,94.0,null,null,null
"""50%""",1.5032392e7,"""2153-08-07 10:00:00""",null,97.0,null,null,null
"""75%""",1.7522306e7,"""2173-10-13 18:30:00""",null,99.0,null,null,null
"""max""",1.9999987e7,"""2214-07-26 16:00:00""","""oxygen_saturation//%""",9.9e6,"""___""","""mimic-iv""","""labevents"""


In [26]:
import polars as pl
from pathlib import Path

new_path = Path("/home/q039tl/OpenICU.example/output/project/workspace/concept/dobutamine_rate/1.0.0/mimic-iv.parquet")
old_path = Path("/home/q039tl/OpenICU.example/output/project/workspace/concept-cur/dobutamine_rate/1.0.0/mimic-iv.parquet")

new = pl.read_parquet(new_path)
old = pl.read_parquet(old_path)

print("old rows:", old.height)
print("new rows:", new.height)

cols_without_text = [c for c in old.columns if c != "text_value"]

old_no_text = old.select(cols_without_text).unique()
new_no_text = new.select(cols_without_text).unique()

only_old_no_text = old_no_text.join(new_no_text, on=cols_without_text, how="anti")
only_new_no_text = new_no_text.join(old_no_text, on=cols_without_text, how="anti")

print("only old without text_value:", only_old_no_text.height)
print("only new without text_value:", only_new_no_text.height)

old rows: 20528
new rows: 20528
only old without text_value: 10264
only new without text_value: 12003


In [27]:
key = ["subject_id", "time", "code", "dataset", "table"]

old_keys = old.select(key).unique()
new_keys = new.select(key).unique()

print("old unique keys:", old_keys.height)
print("new unique keys:", new_keys.height)

only_old_keys = old_keys.join(new_keys, on=key, how="anti")
only_new_keys = new_keys.join(old_keys, on=key, how="anti")

print("keys only old:", only_old_keys.height)
print("keys only new:", only_new_keys.height)

old unique keys: 10264
new unique keys: 12003
keys only old: 0
keys only new: 1739


In [30]:
import polars as pl
from pathlib import Path

root_new = Path("/home/q039tl/OpenICU.example/output/project/workspace/concept")
root_old = Path("/home/q039tl/OpenICU.example/output/project/workspace/concept-cur")

old_o2 = pl.read_parquet(root_old / "oxygen_saturation/1.0.0/mimic-iv.parquet")
new_o2 = pl.read_parquet(root_new / "oxygen_saturation/1.0.0/mimic-iv.parquet")
new_art_o2 = pl.read_parquet(root_new / "arterial_oxygen_saturation/1.0.0/mimic-iv.parquet")

print("old oxygen_saturation:", old_o2.height)
print("new oxygen_saturation:", new_o2.height)
print("new arterial_oxygen_saturation:", new_art_o2.height)
print("new combined:", new_o2.height + new_art_o2.height)

old oxygen_saturation: 9617261
new oxygen_saturation: 9377702
new arterial_oxygen_saturation: 358660
new combined: 9736362


In [31]:
for name, df in [
    ("old_o2", old_o2),
    ("new_o2", new_o2),
    ("new_art_o2", new_art_o2),
]:
    print("\n" + "=" * 80)
    print(name)
    print(
        df.group_by(["table", "code"])
        .len()
        .sort("len", descending=True)
    )


old_o2
shape: (2, 3)
┌─────────────┬──────────────────────┬─────────┐
│ table       ┆ code                 ┆ len     │
│ ---         ┆ ---                  ┆ ---     │
│ str         ┆ str                  ┆ u32     │
╞═════════════╪══════════════════════╪═════════╡
│ chartevents ┆ oxygen_saturation//% ┆ 9377702 │
│ labevents   ┆ oxygen_saturation//% ┆ 239559  │
└─────────────┴──────────────────────┴─────────┘

new_o2
shape: (1, 3)
┌─────────────┬──────────────────────┬─────────┐
│ table       ┆ code                 ┆ len     │
│ ---         ┆ ---                  ┆ ---     │
│ str         ┆ str                  ┆ u32     │
╞═════════════╪══════════════════════╪═════════╡
│ chartevents ┆ oxygen_saturation//% ┆ 9377702 │
└─────────────┴──────────────────────┴─────────┘

new_art_o2
shape: (2, 3)
┌─────────────┬───────────────────────────────┬────────┐
│ table       ┆ code                          ┆ len    │
│ ---         ┆ ---                           ┆ ---    │
│ str         ┆ str     

In [32]:
from pathlib import Path
from collections import defaultdict

from openicu_ricu_converter.loader import load_concept_dict
from openicu_ricu_converter.settings import ConverterSettings, load_project_event_names, load_project_event_names_by_code_column
from openicu_ricu_converter.mapper import RICUToOpenICUMapper

concepts = load_concept_dict(
    Path("/home/q039tl/ricu/inst/extdata/config/concept-dict.json")
)

settings = ConverterSettings.from_file(None)

openicu_config = Path("/home/q039tl/OpenICU/config")

# so wie CLI: OpenICU event names hineinladen
for dataset, table_map in load_project_event_names(openicu_config).items():
    settings.event_names.setdefault(dataset, {}).update(table_map)

inferred_by_code_column = load_project_event_names_by_code_column(openicu_config)

mapper = RICUToOpenICUMapper(
    settings,
    inferred_events_by_code_column=inferred_by_code_column,
)

files = mapper.build_files(
    concepts,
    sources=["miiv", "eicu"],
)

by_path = defaultdict(list)
for f in files:
    by_path[f.path].append(f)

collisions = {path: fs for path, fs in by_path.items() if len(fs) > 1}

print("number of path collisions:", len(collisions))

for path, fs in collisions.items():
    if "oxygen_saturation" in path or "arterial" in path:
        print("\nCOLLISION:", path)
        for f in fs:
            print(f.content)

number of path collisions: 3

COLLISION: concept/respiratory/oxygen_saturation.yml
{'name': 'oxygen_saturation', 'version': '1.0.0', 'unit': '%', 'extension_columns': {'dataset': 'col("dataset")', 'table': 'col("table")'}}
{'name': 'oxygen_saturation', 'version': '1.0.0', 'unit': '%', 'extension_columns': {'dataset': 'col("dataset")', 'table': 'col("table")'}}

COLLISION: dataset/mimic-iv/3.1/concept/oxygen_saturation.yml
{'type': 'simple', 'mappings': [{'pattern': {'table': 'chartevents', 'event': 'CHART', 'code': '(220277(//.*)?|226253(//.*)?|220227(//.*)?)'}, 'columns': {'numeric_value': 'col(numeric_value)', 'text_value': 'col(text_value)'}}, {'pattern': {'table': 'labevents', 'event': 'LAB', 'code': '(50817(//.*)?)'}, 'columns': {'numeric_value': 'col(numeric_value)', 'text_value': 'col(text_value)'}}]}
{'type': 'simple', 'mappings': [{'pattern': {'table': 'chartevents', 'event': 'CHART', 'code': '(220277(//.*)?|226253(//.*)?)'}, 'columns': {'numeric_value': 'col(numeric_value)', 